In [1]:
%load_ext autoreload
%autoreload 2

import os
from openai import OpenAI
from dotenv import load_dotenv
from datasets import load_dataset
from core.rag import default_rag_client
from core.embedding.sentence_transformer_embedder import SentenceTransformerEmbedder
from tqdm.notebook import tqdm

load_dotenv()

True

In [2]:
llm = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

embedder = SentenceTransformerEmbedder()

rag = default_rag_client(
    llm_client=llm,
    embedder=embedder,
    workspace="audio_test",
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path
from tqdm import tqdm

raw_audio_dir = Path("./raw_audio")

audio_files = [p for p in raw_audio_dir.iterdir() if p.is_file()]

for audio_file in tqdm(audio_files, desc="Ingesting audio into RAG"):
    rag.add_file(
        path=str(audio_file),
        file_type=audio_file.suffix.lstrip("."),
        metadata={
            "source": "raw_audio",
            "filename": audio_file.name,
        }
    )

print("Ingestion complete!")

Ingesting documents into RAG:   0%|          | 0/250 [00:00<?, ?it/s]

Ingestion complete!


In [12]:
sample_row = sampled_data[2]
test_query = sample_row["question"] 

print(f"Test Query: \"{test_query}\"\n")
print("Finding top relevant chunks")

relevant_chunks = rag.retrieve_chunks(query=test_query, top_k=30)

for idx, chunk in enumerate(relevant_chunks):
    score = chunk.get('score', 0.0) if isinstance(chunk, dict) else getattr(chunk, 'score', 0.0)
    text = chunk.get('text', '') if isinstance(chunk, dict) else getattr(chunk, 'text', '')
    
    print(f"\n[Chunk {idx+1}] (Similarity Score: {score:.4f})")
    print(text + "...") 

Test Query: "Why cant I load and AEL when using IE 11 JRE 8 Application Blocked by Java Security
"

Finding top relevant chunks

[Chunk 1] (Similarity Score: 0.6980)
turing a Java applet (e.g. AEL).

3. Click through any necessary security warnings.

4. Wait for the applet to load and start running.

5. Attempt to close the applet, for example by clicking the "close" button on its page, or by logging out.

Expected result: the applet closes, the browser continues to function properly.

Actual result: the browser crashes. 


RESOLVING THE PROBLEM
Upgrade to Firefox 38.5 ESR. 

Alternatively downgrade to Firefox 38.3 ESR, or use Internet Explorer....

[Chunk 2] (Similarity Score: 0.6857)
TECHNOTE (TROUBLESHOOTING)

PROBLEM(ABSTRACT)
 Firefox 38.4 ESR crashes when closing a running Java applet in Windows. 
This issue impacts on Web GUI usage for all features involving applets, e.g. AEL, dashboards, maps, map editor, etc. 

ENVIRONMENT
This issue has been reproduced in Windows environments

In [5]:
rag.clear_knowledge_base()
rag.delete_workspace()